# ============================================
# Customer Churn Data Cleaning
# Customer Churn & Retention Analysis
# ============================================

1. Imports
2. Load Dataset
3. Dataset Overview
4. Missing Value Analysis
5. Duplicate Check
6. Data Type Fixes
7. Feature Engineering
8. Churn Encoding
9. Export Cleaned Dataset

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv(
    '../datasets/raw/customer_churn.csv'
)

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 7043
Columns: 21


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [7]:
missing_values = df.isnull().sum()

missing_values

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [8]:
#-- TotalCharges has 11 missing values, but it is an object type. Let's check the unique values to understand the issue.
df['TotalCharges'].unique()

<StringArray>
[  '29.85',  '1889.5',  '108.15', '1840.75',  '151.65',   '820.5',  '1949.4',
   '301.9', '3046.05', '3487.95',
 ...
 '2625.25', '6886.25',  '1495.1',   '743.3',  '1419.4',  '1990.5',  '7362.9',
  '346.45',   '306.6',  '6844.5']
Length: 6531, dtype: str

In [9]:
# -- replace blank spaces with NaN values in TotalCharges column
df['TotalCharges'] = df['TotalCharges'].replace(
    ' ',
    np.nan
)

In [10]:
#-- convert TotalCharges to numeric type
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges']
)

In [11]:
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [12]:
#-- drop rows with missing values in TotalCharges column
df = df.dropna()

In [13]:
duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

Duplicate Rows: 0


In [14]:
df = df.drop_duplicates()

## FEATURE ENGINEERING SECTION

In [15]:
# -- create tenure groups based on tenure column
def tenure_group(tenure):

    if tenure <= 12:
        return '0-1 Year'

    elif tenure <= 24:
        return '1-2 Years'

    elif tenure <= 48:
        return '2-4 Years'

    else:
        return '4+ Years'

In [16]:
#-- apply tenure_group function to tenure column and create a new column called tenure_group
df['tenure_group'] = df['tenure'].apply(
    tenure_group
)

In [17]:
# -- create monthly charge groups based on MonthlyCharges column
def monthly_charge_group(charge):

    if charge < 35:
        return 'Low'

    elif charge < 70:
        return 'Medium'

    else:
        return 'High'

In [19]:
#-- apply monthly_charge_group function to MonthlyCharges column and create a new column called monthly_charge_group
df['monthly_charge_group'] = (
    df['MonthlyCharges']
    .apply(monthly_charge_group)
)

## CHURN ENCODING

In [20]:
#-- encode Churn column to binary values (Yes: 1, No: 0) and create a new column called Churn_Encoded
df['Churn_Encoded'] = df['Churn'].map({
    'Yes': 1,
    'No': 0
})

In [21]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group,monthly_charge_group,Churn_Encoded
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-1 Year,Low,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,2-4 Years,Medium,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-1 Year,Medium,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,2-4 Years,Medium,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-1 Year,High,1


In [22]:
df.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customerID            7032 non-null   str    
 1   gender                7032 non-null   str    
 2   SeniorCitizen         7032 non-null   int64  
 3   Partner               7032 non-null   str    
 4   Dependents            7032 non-null   str    
 5   tenure                7032 non-null   int64  
 6   PhoneService          7032 non-null   str    
 7   MultipleLines         7032 non-null   str    
 8   InternetService       7032 non-null   str    
 9   OnlineSecurity        7032 non-null   str    
 10  OnlineBackup          7032 non-null   str    
 11  DeviceProtection      7032 non-null   str    
 12  TechSupport           7032 non-null   str    
 13  StreamingTV           7032 non-null   str    
 14  StreamingMovies       7032 non-null   str    
 15  Contract              7032 non-null  

In [23]:
df.to_csv(
    '../datasets/cleaned/cleaned_churn_data.csv',
    index=False
)